# [7.5] Predictive Concept Decoders - Exercises

Predictive Concept Decoders answer behavioral questions through a sparse concept bottleneck. The local course target is deliberately scoped: build a toy PCD contract, then inspect a pinned `gelu-1l` CUDA preflight over surface-vs-motion residuals.

```text
residual activation -> sparse concepts -> question-conditioned slots -> answer logits
```

A section-ready result is not just green tests. You should be able to explain why a question-agnostic probe fails, why shuffled questions should hurt, and why top-concept removal needs a low-margin active control.


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch as t
import torch.nn.functional as F

chapter = "chapter7_activation_to_language"
section = "part5_predictive_concept_decoders"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_predictive_concept_decoders.tests as tests

GT_TIER = "GT-3"
EXERCISE_ID = "7_5_predictive_concept_decoders"
EXPECTED_RUNTIME = "25-45 minutes for exercises; about 1 minute for the CUDA preflight"
REQUIRES_GPU = True


## Behavioral Question Batches

A PCD row combines an activation, a behavioral question, and an answer label. Keep those fields aligned before doing any concept work.

> ```yaml
> Difficulty: easy
> Importance: high
> You should spend 5-10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_build_pcd_question_batch_validates_shapes_and_questions` passed!
All tests in `test_build_pcd_question_batch_rejects_empty_and_bad_question_ids` passed!
```

The default question bank should contain four questions: Paris, refusal, number-token, and hidden-even.

</details>

<details>
<summary>Help - why store readable question text?</summary>

The decoder only needs ids or embeddings, but the audit needs stable text. Otherwise a report can say `question_id=2` without letting a reader inspect what behavior was asked.

</details>

<details>
<summary>Common bug</summary>

Do not accept empty batches or out-of-range question ids. Those failures should happen at the boundary, not later inside a reduction or embedding lookup.

</details>

<details>
<summary>Solution</summary>

Validate rank, nonempty batch size, id shape, question-bank size, and id range. Convert ids to `long` and return immutable question text.

</details>


In [ ]:
@dataclass(frozen=True)
class PCDQuestionBatch:
    activations: t.Tensor
    question_ids: t.Tensor
    answer_ids: t.Tensor
    question_texts: tuple[str, ...]


def default_pcd_questions() -> tuple[str, ...]:
    raise NotImplementedError()


def build_pcd_question_batch(
    activations: t.Tensor,
    question_ids: t.Tensor,
    answer_ids: t.Tensor,
    question_texts: tuple[str, ...] | None = None,
) -> PCDQuestionBatch:
    raise NotImplementedError()


tests.test_build_pcd_question_batch_validates_shapes_and_questions(
    build_pcd_question_batch,
    default_pcd_questions,
)
tests.test_build_pcd_question_batch_rejects_empty_and_bad_question_ids(
    build_pcd_question_batch,
)


## Sparse Concept Encoding

Project activations onto concept directions, discard negative evidence, optionally threshold weak positives, optionally keep top-k concepts, then report density.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10-15 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_sparse_concept_encode_and_sparsity_controls` passed!
All tests in `test_sparse_concept_encode_and_sparsity_reject_bad_controls` passed!
```

The identity fixture should have one active concept per row and density `1/3`.

</details>

<details>
<summary>Help - why top-k after thresholding?</summary>

Top-k is a sparsity control, not a sign correction. ReLU and threshold first, then keep the strongest remaining concepts.

</details>

<details>
<summary>Common bug</summary>

The concept directions are columns with shape `[d_model, n_concepts]`. Multiplying by `concept_directions.T` silently gives the wrong contract.

</details>

<details>
<summary>Solution</summary>

Use `activations.float() @ concept_directions.float()`, add an optional bias, apply ReLU, threshold, then scatter a top-k mask. Report density over all entries.

</details>


In [ ]:
@dataclass(frozen=True)
class ConceptSparsityReport:
    mean_l0: float
    density: float
    passes_sparsity: bool


def sparse_concept_encode(
    activations: t.Tensor,
    concept_directions: t.Tensor,
    *,
    bias: t.Tensor | None = None,
    top_k: int | None = None,
    threshold: float = 0.0,
) -> t.Tensor:
    raise NotImplementedError()


def concept_sparsity_report(
    concepts: t.Tensor,
    *,
    active_threshold: float = 0.0,
    max_density: float = 0.3,
) -> ConceptSparsityReport:
    raise NotImplementedError()


tests.test_sparse_concept_encode_and_sparsity_controls(
    sparse_concept_encode,
    concept_sparsity_report,
)
tests.test_sparse_concept_encode_and_sparsity_reject_bad_controls(
    sparse_concept_encode,
    concept_sparsity_report,
)


## Question-Conditioned Features

The PCD needs explicit concept-question interaction features. Allocate one concept slot per question, then decode from those slots plus a question embedding.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_question_conditioned_decoder_uses_question_information` passed!
All tests in `test_question_conditioned_decoder_respects_arbitrary_weight_and_bias` passed!
All tests in `test_question_conditioned_features_and_decoder_reject_bad_shapes` passed!
```

For two concepts and two questions, the expanded feature width should be `4`.

</details>

<details>
<summary>Help - what should be nonzero?</summary>

Only the slot for the current question should contain the row's sparse concepts. All other question slots should be zero for that row.

</details>

<details>
<summary>Common bug</summary>

Concatenating the question embedding is not the same as concept-question interaction. You need the slot expansion before the decoder.

</details>

<details>
<summary>Solution</summary>

Build a zero tensor of shape `[examples, question_count * n_concepts]`, compute row-specific offsets from `question_ids`, and scatter concept values into those slots.

</details>


In [ ]:
def question_conditioned_concept_features(
    concepts: t.Tensor,
    question_ids: t.Tensor,
    question_count: int,
) -> t.Tensor:
    raise NotImplementedError()


def question_conditioned_decoder_logits(
    concepts: t.Tensor,
    question_embeddings: t.Tensor,
    decoder_weight: t.Tensor,
    *,
    decoder_bias: t.Tensor | None = None,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_question_conditioned_decoder_uses_question_information(
    question_conditioned_decoder_logits,
)
tests.test_question_conditioned_decoder_respects_arbitrary_weight_and_bias(
    question_conditioned_decoder_logits,
)
tests.test_question_conditioned_features_and_decoder_reject_bad_shapes(
    question_conditioned_concept_features,
    question_conditioned_decoder_logits,
)


## Train The Question-Conditioned Decoder

Now fit the decoder. The fixture repeats the same concepts under different questions, so a question-agnostic probe cannot solve all rows.

> ```yaml
> Difficulty: hard
> Importance: high
> You should spend 15-20 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_trained_question_conditioned_decoder_learns_concept_question_interaction` passed!
All tests in `test_train_question_conditioned_decoder_rejects_empty_and_misaligned_batches` passed!
```

The trained decoder should reach `train_accuracy = 1.0` and predict `[1, 0, 0, 1]` on the toy rows.

</details>

<details>
<summary>Help - why a linear head can work</summary>

The interaction is already represented by the feature expansion. After that, a linear decoder can assign different weights to the same concept under different questions.

</details>

<details>
<summary>Common bug</summary>

Do not train on raw concepts when the test expects expanded interaction features. That turns the exercise back into a probe.

</details>

<details>
<summary>Solution</summary>

Seed PyTorch, initialize a small `[feature_width + question_width, 2]` decoder and bias, train with cross-entropy, then return detached weights plus a report containing final loss, accuracy, steps, and seed.

</details>


In [ ]:
@dataclass(frozen=True)
class DecoderTrainingReport:
    final_loss: float
    train_accuracy: float
    steps: int
    seed: int


def _prediction_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def train_question_conditioned_decoder(
    concepts: t.Tensor,
    question_embeddings: t.Tensor,
    answer_ids: t.Tensor,
    *,
    steps: int = 400,
    lr: float = 0.08,
    seed: int = 0,
    weight_decay: float = 0.0,
) -> tuple[t.Tensor, t.Tensor, DecoderTrainingReport]:
    raise NotImplementedError()


tests.test_trained_question_conditioned_decoder_learns_concept_question_interaction(
    question_conditioned_concept_features,
    train_question_conditioned_decoder,
    question_conditioned_decoder_logits,
)
tests.test_train_question_conditioned_decoder_rejects_empty_and_misaligned_batches(
    train_question_conditioned_decoder,
)


## Baseline Comparison

A PCD earns its keep only if the concept-question decoder beats simpler baselines. This fixture keeps a question-agnostic probe at chance.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_pcd_comparison_report_beats_baselines` passed!
All tests in `test_pcd_comparison_report_scores_each_baseline_independently` passed!
```

Expected toy accuracies: PCD `1.0`, probe `0.5`, SAE-style classifier `0.5`, oracle-style fixture `0.75`.

</details>

<details>
<summary>Help - what does this prove?</summary>

It proves a local compositional advantage over the listed baselines. It does not prove broad semantic coverage or arbitrary generalization.

</details>

<details>
<summary>Common bug</summary>

A tie with the best baseline is not success. The local report requires `pcd_accuracy > best_baseline_accuracy`.

</details>

<details>
<summary>Solution</summary>

Score every logits tensor independently with `argmax`, take the maximum baseline accuracy, then set both comparison booleans from strict greater-than checks.

</details>


In [ ]:
@dataclass(frozen=True)
class PCDComparisonReport:
    pcd_accuracy: float
    probe_accuracy: float
    sae_classifier_accuracy: float
    activation_oracle_accuracy: float
    best_baseline_accuracy: float
    beats_probe: bool
    beats_best_baseline: bool


def pcd_comparison_report(
    pcd_logits: t.Tensor,
    probe_logits: t.Tensor,
    sae_classifier_logits: t.Tensor,
    activation_oracle_logits: t.Tensor,
    answer_ids: t.Tensor,
) -> PCDComparisonReport:
    raise NotImplementedError()


tests.test_pcd_comparison_report_beats_baselines(pcd_comparison_report)
tests.test_pcd_comparison_report_scores_each_baseline_independently(
    pcd_comparison_report,
)


## Concept Audit Controls

Stable, useful concepts should survive seed changes, matter more than a low-margin active-control removal, and have names that make the selected cluster inspectable.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 15-20 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_concept_stability_removal_and_audit_controls` passed!
All tests in `test_concept_audit_controls_reject_bad_inputs` passed!
```

The toy audit should select concept ids `(1, 2)` with names containing `refusal`.

</details>

<details>
<summary>Help - interpreting removal controls</summary>

This is decoder-level evidence. It compares removing the top active concept against removing a lower-margin active concept from the same row.

</details>

<details>
<summary>Common bug</summary>

Do not measure removal delta on the newly predicted answer. Measure the original target-answer logit before and after removal.

</details>

<details>
<summary>Solution</summary>

Use top-k set Jaccard for stability, target-logit deltas for removal, and simple substring matching only as an audit aid for concept names.

</details>


In [ ]:
@dataclass(frozen=True)
class ConceptStabilityReport:
    top_concepts_by_seed: tuple[tuple[int, ...], ...]
    mean_pairwise_jaccard: float
    stable: bool


@dataclass(frozen=True)
class ConceptRemovalReport:
    original_answer: int
    top_removed_answer: int
    random_removed_answer: int
    top_removal_changed: bool
    random_removal_changed: bool
    top_removal_delta: float
    random_removal_delta: float
    random_removal_does_less: bool


@dataclass(frozen=True)
class ConceptAuditReport:
    selected_concept_ids: tuple[int, ...]
    selected_concept_names: tuple[str, ...]
    explanation: str
    names_expected_cluster: bool


def concept_stability_report(
    concept_scores_by_seed: list[t.Tensor],
    *,
    top_k: int = 3,
    min_jaccard: float = 0.5,
) -> ConceptStabilityReport:
    raise NotImplementedError()


def concept_removal_report(
    original_logits: t.Tensor,
    top_removed_logits: t.Tensor,
    random_removed_logits: t.Tensor,
    *,
    target_answer_id: int | None = None,
) -> ConceptRemovalReport:
    raise NotImplementedError()


def concept_audit_report(
    concept_scores: t.Tensor,
    concept_names: list[str],
    expected_cluster_terms: list[str],
    *,
    top_k: int = 2,
) -> ConceptAuditReport:
    raise NotImplementedError()


tests.test_concept_stability_removal_and_audit_controls(
    concept_stability_report,
    concept_removal_report,
    concept_audit_report,
)
tests.test_concept_audit_controls_reject_bad_inputs(
    concept_stability_report,
    concept_removal_report,
    concept_audit_report,
)


## Combined Contract

Compose a CPU-only report before touching the live model. It should contain sparse concepts, trained decoder evidence, baseline comparison, stability, removal, and concept-name audit.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

The contract should expose trained decoder predictions `[1, 0, 0, 1]`, `pcd_accuracy = 1.0`, `probe_accuracy = 0.5`, and `random_removal_does_less = True`.

</details>

<details>
<summary>Help - reading the combined contract</summary>

The smoke report is not the final evidence. It is a fast check that your implementation composes the same evidence types as the CUDA report.

</details>

<details>
<summary>Common bug</summary>

Do not return only `preflight_passed`. The interesting fields are the metric controls that make the pass meaningful.

</details>

<details>
<summary>Solution</summary>

Use the helper functions above. Make the report nested and explicit so a reader can inspect each gate without rerunning the full CUDA preflight.

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)


## Signature Result

The committed CUDA report is the section-scale result. It should show the PCD beating baselines, question shuffling failing, seed-stable top concepts, and top removal doing more damage than the low-margin active control.

<details>
<summary>Expected output</summary>

```text
pcd_accuracy: 1.0
probe_accuracy: 0.5
best_baseline_accuracy: 0.5
question_shuffle_accuracy: 0.0
pcd_seed_min_accuracy: 1.0
top_removal_changed: True
random_removal_does_less: True
peak_vram_gb: below 24.0
```

</details>

<details>
<summary>Help - why inspect the report before live CUDA?</summary>

The report is the committed evidence artifact for review. The live run checks reproducibility on this machine; the report check verifies that the artifact contains the claimed gates.

</details>

<details>
<summary>Common bug</summary>

Do not call this a full Activation Oracle comparison. In this section the legacy `activation_oracle_accuracy` field is a non-interaction activation+question baseline.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["preflight_passed"]
    assert gpu["within_vram_budget"]
    assert gpu["pcd_accuracy"] == 1.0
    assert gpu["probe_accuracy"] == 0.5
    assert gpu["best_baseline_accuracy"] <= 0.75
    assert gpu["beats_best_baseline"]
    assert gpu["question_shuffle_accuracy"] <= 0.75
    assert gpu["pcd_seed_min_accuracy"] == 1.0
    assert gpu["stable"]
    assert gpu["top_removal_changed"]
    assert gpu["random_removal_does_less"]
    assert gpu["random_removed_concept_active"]
    assert gpu["names_expected_cluster"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_full_experiment(max_vram_gb=24.0)
{
    "pcd_accuracy": gpu["pcd_accuracy"],
    "probe_accuracy": gpu["probe_accuracy"],
    "best_baseline_accuracy": gpu["best_baseline_accuracy"],
    "question_shuffle_accuracy": gpu["question_shuffle_accuracy"],
    "top_removal_delta": round(gpu["top_removal_delta"], 4),
    "random_removal_delta": round(gpu["random_removal_delta"], 4),
    "peak_vram_gb": round(gpu["peak_vram_gb"], 3),
}


## Limitations

This is a GT-3 local mini-PCD preflight on one pinned `gelu-1l` hook and a tiny safe surface-vs-motion split. It is not a broad PCD benchmark, not downstream model-level causal proof, not semantic proof from names, and not a full Activation Oracle comparison.

## Further Research

Scale the question bank, replace handcrafted concept directions with learned dictionaries or SAEs, run true model-level ablations, add more seeds and layers, and keep the same baseline/shuffle/removal controls as the task grows.
